# Evaluation and Figures

Run after all three methods have been trained with `02_train.ipynb`.
Loads everything from Google Drive; generates oracle reward, perplexity, and Figures 1–3.

If you downloaded the model zips locally, unzip them into `checkpoints/<method>/final/`
on Drive before running this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q 'transformers>=4.40' datasets matplotlib

In [ ]:
import os, json
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2ForSequenceClassification
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

ROOT       = '/content/drive/MyDrive/cal-simpo-outputs'
REWARD_DIR = f'{ROOT}/reward_model'
SFT_DIR    = f'{ROOT}/sft_model'
PREF_DIR   = f'{ROOT}/pref_data'
CKPT_DIR   = f'{ROOT}/checkpoints'
RES_DIR    = f'{ROOT}/results'
FIGS_DIR   = f'{ROOT}/figures'
for d in [RES_DIR, FIGS_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
PROMPT_LEN = 32
MAX_NEW    = 128
SEED       = 42
DPO_BETA   = 0.001
SIMPO_BETA = 2.0

tok = AutoTokenizer.from_pretrained(SFT_DIR)
tok.pad_token = tok.eos_token
print(f'Device: {DEVICE}')

## Step 5 — Evaluation

In [ ]:
# Load oracle reward model and frozen small GPT-2 for perplexity
reward_model = GPT2ForSequenceClassification.from_pretrained(REWARD_DIR).to(DEVICE)
reward_model.eval()

ppl_model = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)  # small, never fine-tuned
ppl_model.eval()

@torch.no_grad()
def oracle_reward(texts, batch_size=16):
    parts = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i+batch_size], truncation=True, max_length=512,
                  padding=True, return_tensors='pt').to(DEVICE)
        lp  = F.log_softmax(reward_model(**enc).logits, dim=-1)
        parts.append((lp[:, 1] - lp[:, 0]).cpu())
    return torch.cat(parts)

@torch.no_grad()
def compute_ppl(texts, max_len=256, batch=8):
    ppls = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i+batch], truncation=True, max_length=max_len,
                  padding=True, return_tensors='pt').to(DEVICE)
        out = ppl_model(**enc)
        lp  = F.log_softmax(out.logits[:, :-1], dim=-1)
        tgt = enc.input_ids[:, 1:]
        msk = enc.attention_mask[:, 1:].float()
        nll = -(lp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1) * msk).sum(-1) / msk.sum(-1).clamp(1)
        ppls.extend(torch.exp(nll).tolist())
    return float(np.mean(ppls))

# Held-out prompts (different seed from preference data construction)
imdb      = load_dataset('imdb')
rng_e     = np.random.default_rng(SEED + 1)
eval_prompts = [
    tok.decode(tok.encode(imdb['test'][int(i)]['text'],
               add_special_tokens=False)[:PROMPT_LEN])
    for i in rng_e.permutation(len(imdb['test']))[:200]
]
print(f'Evaluation prompts: {len(eval_prompts)}')

In [ ]:
@torch.no_grad()
def eval_policy(name, model_dir):
    """Generate 200 completions; return oracle reward + perplexity."""
    if not os.path.isdir(model_dir):
        print(f'  Skipping {name} — checkpoint not found at {model_dir}')
        return None
    model = GPT2LMHeadModel.from_pretrained(model_dir).to(DEVICE)
    model.eval()
    texts = []
    for p in tqdm(eval_prompts, desc=name, leave=False):
        ids = tok(p, return_tensors='pt').input_ids.to(DEVICE)
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=True,
                              temperature=0.7, pad_token_id=tok.eos_token_id)
        texts.append(p + tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    del model;  torch.cuda.empty_cache()
    r = oracle_reward(texts)
    return {'method': name, 'reward_mean': float(r.mean()),
            'reward_std': float(r.std()), 'perplexity': compute_ppl(texts)}

eval_results = []
for name, d in [
    ('SFT',    SFT_DIR),
    ('DPO',    f'{CKPT_DIR}/dpo/final'),
    ('Cal-DPO',f'{CKPT_DIR}/caldpo/final'),
    ('SimPO',  f'{CKPT_DIR}/simpo/final'),
]:
    result = eval_policy(name, d)
    if result:
        eval_results.append(result)

json.dump(eval_results, open(f'{RES_DIR}/eval_results.json', 'w'), indent=2)

print(f"\n{'Method':<10} {'Oracle Reward':>18} {'Perplexity':>12}")
print('-' * 44)
for r in eval_results:
    print(f"{r['method']:<10} {r['reward_mean']:>+7.4f} ± {r['reward_std']:<7.4f} {r['perplexity']:>12.2f}")

## Step 6 — Figures

In [ ]:
COLORS = {'DPO': '#1f77b4', 'Cal-DPO': '#2ca02c', 'SimPO': '#ff7f0e', 'SFT': '#9467bd'}

def load_log(method):
    path = f'{RES_DIR}/{method}_log.json'
    return json.load(open(path)) if os.path.exists(path) else []

def reward_series(log):
    """Extract eval-step reward series from trl log_history."""
    # trl prefixes eval metrics with 'eval_'
    entries = [e for e in log if 'eval_rewards/chosen' in e]
    if not entries:
        return [], [], [], []
    return (
        [e['step']                      for e in entries],
        [e['eval_rewards/chosen']        for e in entries],
        [e['eval_rewards/rejected']      for e in entries],
        [e['eval_rewards/margins']       for e in entries],
    )

# ── Figure 1: Training Dynamics ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (label, logf) in zip(axes, [('DPO','dpo'), ('Cal-DPO','caldpo'), ('SimPO','simpo')]):
    steps, ch, rj, mg = reward_series(load_log(logf))
    if steps:
        ax.plot(steps, ch, 'b-',  lw=1.8, label='Chosen')
        ax.plot(steps, rj, 'r-',  lw=1.8, label='Rejected')
        ax.plot(steps, mg, 'g--', lw=1.8, label='Margin')
        ax.axhline(0, color='k', lw=0.8, ls=':', alpha=0.5)
        ax.legend(fontsize=9)
    else:
        ax.text(0.5, 0.5, 'no log data', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_xlabel('Training Steps');  ax.set_ylabel('Implicit Reward')
    ax.grid(alpha=0.3)
fig.suptitle('Figure 1: Training Dynamics', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/fig1_dynamics.png', dpi=150, bbox_inches='tight');  plt.show()

# ── Figure 2: Final Performance ───────────────────────────────────────────────
res   = json.load(open(f'{RES_DIR}/eval_results.json'))
names = [r['method'] for r in res]
rwds  = [r['reward_mean'] for r in res]
stds  = [r['reward_std']  for r in res]
ppls  = [r['perplexity']  for r in res]
cols  = [COLORS.get(m, '#555') for m in names]
x     = np.arange(len(names))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
a1.bar(x, rwds, yerr=stds, capsize=5, color=cols, alpha=0.85)
a1.set_xticks(x);  a1.set_xticklabels(names)
a1.set_ylabel('Oracle Reward Score');  a1.set_title('Oracle Reward (↑ better)')
a1.grid(axis='y', alpha=0.3)
a2.bar(x, ppls, color=cols, alpha=0.85)
a2.set_xticks(x);  a2.set_xticklabels(names)
a2.set_ylabel('Perplexity');  a2.set_title('Perplexity (↓ better)')
a2.grid(axis='y', alpha=0.3)
fig.suptitle('Figure 2: Final Performance Comparison', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/fig2_performance.png', dpi=150, bbox_inches='tight');  plt.show()

# ── Figure 3: Training Loss Curves ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for label, logf in [('DPO','dpo'), ('Cal-DPO','caldpo'), ('SimPO','simpo')]:
    entries = [e for e in load_log(logf) if 'loss' in e and 'eval_loss' not in e]
    if entries:
        ax.plot([e['step'] for e in entries], [e['loss'] for e in entries],
                label=label, color=COLORS[label], lw=1.8)
ax.set_xlabel('Steps');  ax.set_ylabel('Training Loss')
ax.set_title('Figure 3: Training Loss Curves')
ax.legend();  ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/fig3_loss.png', dpi=150, bbox_inches='tight');  plt.show()
print(f'Figures saved to {FIGS_DIR}')

In [ ]:
import shutil, os
from google.colab import files

# Copy only results/ and figures/ into a temp folder, then zip and download
tmp = '/content/download_tmp'
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(RES_DIR,  f'{tmp}/results')
shutil.copytree(FIGS_DIR, f'{tmp}/figures')
shutil.make_archive('/content/results_and_figs', 'zip', tmp)
files.download('/content/results_and_figs.zip')
print('Downloading results_and_figs.zip (results + figures only, no model weights)')